In [133]:
import pandas as pd
import os
import rasterio
import tqdm
import numpy as np
import re
import rasterio
import tqdm
import geopandas as gpd

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\CB')

# Generate dummy IDs

In [4]:
file_path = "original.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def infer_format(series):
    non_null = series.dropna().astype(str)
    if non_null.empty:
        return ("ID", 4)
    sample = non_null.iloc[0]
    m = re.match(r"^([A-Za-z]*)(\d+)$", sample)
    if m:
        prefix = m.group(1)
        width = len(m.group(2))
    else:
        prefix = ""
        width = max(len(re.findall(r"\d+", s)[-1]) if re.findall(r"\d+", s) else 4 for s in non_null)
    return (prefix, width)

def next_available(existing_nums, start=1):
    n = start
    while n in existing_nums:
        n += 1
    return n

def fill_ids(df, id_col, base_col):
    if id_col not in df.columns or base_col not in df.columns:
        return df

    # Ensure ID column can hold string values without dtype errors
    df[id_col] = df[id_col].astype("string")

    prefix, width = infer_format(df[id_col])
    existing = df[id_col].dropna().astype(str)

    def extract_num(x):
        m = re.match(r"^[A-Za-z]*?(\d+)$", x)
        return int(m.group(1)) if m else None

    used_nums = {n for n in (extract_num(x) for x in existing) if n is not None}

    # map base value -> existing id
    base_to_id = (
        df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
        .astype(str)
        .to_dict()
    )

    missing_mask = df[id_col].isna()
    for idx, row in df[missing_mask].iterrows():
        base_val = row[base_col]
        if pd.isna(base_val):
            continue
        if base_val in base_to_id:
            df.at[idx, id_col] = base_to_id[base_val]
        else:
            new_num = next_available(used_nums)
            used_nums.add(new_num)
            new_id = f"{prefix}{str(new_num).zfill(width)}"
            df.at[idx, id_col] = new_id
            base_to_id[base_val] = new_id

    # Remove trailing .0 from IDs that were read as floats
    df[id_col] = df[id_col].astype("string").str.replace(r"\.0$", "", regex=True)
    return df

biota = fill_ids(biota, "StationID", "Station")
biota = fill_ids(biota, "DeviceID", "Device")
biota = fill_ids(biota, "SampleID", "Sample")
biota = fill_ids(biota, "SubSampleID", "SubSample")
biota = fill_ids(biota, "SubsetID", "Subset")
biota = fill_ids(biota, "BiotaID", "Biota")

sheets["Biota"] = biota

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

In [6]:
# reload workbook
sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def fill_ids_from_biota(df, biota_df, id_col, base_col):
    if df is None or biota_df is None:
        return df
    if id_col not in df.columns or base_col not in df.columns:
        return df

    mapping = (
        biota_df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
    )
    df[id_col] = df[id_col].fillna(df[base_col].map(mapping))
    return df

sheet_id_map = {
    "Station": [("StationID", "Station")],
    "Device": [("StationID", "Station"), ("DeviceID", "Device")],
    "Sample": [("StationID", "Station"), ("DeviceID", "Device"), ("SampleID", "Sample")],
    "Subsample": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
    ],
    "Subset": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
        ("SubsetID", "Subset"),
    ],
}

for sheet_name, mappings in sheet_id_map.items():
    if sheet_name in sheets:
        df_sheet = sheets[sheet_name]
        for id_col, base_col in mappings:
            df_sheet = fill_ids_from_biota(df_sheet, biota, id_col, base_col)
        sheets[sheet_name] = df_sheet

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

# Create long format

In [188]:
biota = pd.read_excel(file_path, sheet_name="Biota")
biota.to_parquet("cb_long.parquet", index=False)

# Key in necessary columns

Add Project column

In [189]:
biota = pd.read_parquet("cb_long.parquet")
subset_df = sheets.get("Subset", pd.DataFrame())

if "SubsetID" in biota.columns and "SubsetID" in subset_df.columns and "Project" in subset_df.columns:
    subset_proj = subset_df[["SubsetID", "Project"]].drop_duplicates(subset=["SubsetID"])
    biota = biota.merge(subset_proj, on="SubsetID", how="left", suffixes=("", "_subset"))
    if "Project_subset" in biota.columns:
        biota["Project"] = biota["Project"].fillna(biota["Project_subset"])
        biota = biota.drop(columns=["Project_subset"])
    biota.to_parquet("cb_long.parquet", index=False)

Add Title and Realm columns

In [190]:
biota = pd.read_parquet("cb_long.parquet")
dataset_df = sheets.get("Dataset", pd.DataFrame())
subset_df = sheets.get("Subset", pd.DataFrame())

if not dataset_df.empty and "Name" in dataset_df.columns and "Title" in dataset_df.columns and "Realm" in dataset_df.columns:
    project_to_meta = (
        dataset_df[["Name", "Title", "Realm"]]
        .dropna(subset=["Name"])
        .drop_duplicates(subset=["Name"])
        .set_index("Name")
    )

    subset_meta = subset_df.copy()
    subset_meta["Title"] = subset_meta["Project"].map(project_to_meta["Title"])
    subset_meta["Realm"] = subset_meta["Project"].map(project_to_meta["Realm"])

    subset_meta = subset_meta[["SubsetID", "Title", "Realm"]].drop_duplicates(subset=["SubsetID"])

    biota = biota.merge(subset_meta, on="SubsetID", how="left", suffixes=("", "_subset"))
    if "Title_subset" in biota.columns:
        biota["Title"] = biota["Title"].fillna(biota["Title_subset"])
        biota = biota.drop(columns=["Title_subset"])
    if "Realm_subset" in biota.columns:
        biota["Realm"] = biota["Realm"].fillna(biota["Realm_subset"])
        biota = biota.drop(columns=["Realm_subset"])

    biota.to_parquet("cb_long.parquet", index=False)

Add coordinates, depth, dates

In [191]:
biota = pd.read_parquet("cb_long.parquet")
station_df = pd.read_excel(file_path, sheet_name="Station")

cols = [
    "StationID",
    "LatStart", "LonStart", "LatEnd", "LonEnd",
    "DepthStart", "DepthEnd",
    "DateStart", "DateEnd",
]
station_coords = station_df[[c for c in cols if c in station_df.columns]].drop_duplicates(subset=["StationID"])

# Drop columns from biota that will come from station_coords (except StationID)
cols_to_drop = [c for c in cols[1:] if c in biota.columns and c in station_coords.columns]
if cols_to_drop:
    biota = biota.drop(columns=cols_to_drop)

biota = biota.merge(station_coords, on="StationID", how="left")

def preserve_date_value(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).isoformat()
    return str(value)

for col in ["DateStart", "DateEnd"]:
    if col in biota.columns:
        biota[col] = biota[col].map(preserve_date_value).astype("string")

biota.to_parquet("cb_long.parquet", index=False)


In [192]:
cb = pd.read_parquet("cb_long.parquet")

# Extract year from DateStart column
if "DateStart" in cb.columns:
    cb["year"] = pd.to_datetime(cb["DateStart"], errors="coerce").dt.year

cb.to_parquet("cb_long.parquet", index=False)
print("Added year column to cb_long.parquet")

Added year column to cb_long.parquet


Sample Remark

In [ ]:
cb = pd.read_parquet("cb_long.parquet")
sample_df = pd.read_excel(file_path, sheet_name="Sample")

if "SampleID" in cb.columns and "SampleID" in sample_df.columns and "Remark" in sample_df.columns:
    sample_remark = sample_df[["SampleID", "Remark"]].drop_duplicates(subset=["SampleID"])
    sample_remark = sample_remark.rename(columns={"Remark": "SampleRemark"})
    cb = cb.merge(sample_remark, on="SampleID", how="left")

cb.to_parquet("cb_long.parquet", index=False)

Add taxonomic details

In [194]:
biota = pd.read_parquet("cb_long.parquet")
taxon_df = sheets.get("Taxon", pd.DataFrame())
biota_key = "AphiaId (Accepted)"
taxon_key = "ValidAphiaId"
cols_to_add = [
    taxon_key,
    "Status",
    "UnacceptReason",
    "Rank",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "ValidAuthority",
]

if not taxon_df.empty and taxon_key in taxon_df.columns and biota_key in biota.columns:
    taxon_meta = (
        taxon_df[[c for c in cols_to_add if c in taxon_df.columns]]
        .dropna(subset=[taxon_key])
        .drop_duplicates(subset=[taxon_key])
    )
    biota = biota.merge(
        taxon_meta,
        left_on=biota_key,
        right_on=taxon_key,
        how="left",
        suffixes=("", "_taxon"),
    )
    for col in cols_to_add[1:]:
        taxon_col = f"{col}_taxon"
        if taxon_col in biota.columns:
            if col in biota.columns:
                biota[col] = biota[col].fillna(biota[taxon_col])
            else:
                biota[col] = biota[taxon_col]
            biota = biota.drop(columns=[taxon_col])
    if taxon_key not in (biota_key,) and taxon_key in biota.columns:
        biota = biota.drop(columns=[taxon_key])

biota.to_parquet("cb_long.parquet", index=False)

Gear

In [195]:
biota = pd.read_parquet("cb_long.parquet")
device_df = sheets.get("Device", pd.DataFrame())

if "DeviceID" in biota.columns and "DeviceID" in device_df.columns and "Gear" in device_df.columns:
    device_gear = device_df[["DeviceID", "Gear"]].drop_duplicates(subset=["DeviceID"])
    biota = biota.merge(device_gear, on="DeviceID", how="left", suffixes=("", "_device"))
    if "Gear_device" in biota.columns:
        biota["Gear"] = biota["Gear"].fillna(biota["Gear_device"]) if "Gear" in biota.columns else biota["Gear_device"]
        biota = biota.drop(columns=["Gear_device"])
    biota.to_parquet("cb_long.parquet", index=False)

In [196]:
biota = pd.read_parquet("cb_long.parquet")
gear_df = sheets.get("Gear", pd.DataFrame())
device_df = sheets.get("Device", pd.DataFrame())

if not gear_df.empty and "Name" in gear_df.columns and "Gear" in device_df.columns:
    gear_cols = [c for c in ["Name", "Category", "Description", "MeshSize"] if c in gear_df.columns]
    gear_lookup = gear_df[gear_cols].drop_duplicates(subset=["Name"])

    device_gear_meta = (
        device_df[["DeviceID", "Gear"]]
        .dropna(subset=["DeviceID", "Gear"])
        .merge(gear_lookup, left_on="Gear", right_on="Name", how="left")
        .drop(columns=["Gear"])
        .drop_duplicates(subset=["DeviceID"])
    )

    biota = biota.merge(device_gear_meta, on="DeviceID", how="left", suffixes=("", "_gear"))
    for col in gear_cols:
        if col in biota.columns and f"{col}_gear" in biota.columns:
            biota[col] = biota[col].fillna(biota[f"{col}_gear"])
            biota = biota.drop(columns=[f"{col}_gear"])
        elif f"{col}_gear" in biota.columns:
            biota[col] = biota[f"{col}_gear"]
            biota = biota.drop(columns=[f"{col}_gear"])

    biota.to_parquet("cb_long.parquet", index=False)

individualCount

In [197]:
biota = pd.read_parquet("cb_long.parquet")

value_cols = [c for c in ["Count", "Abundance"] if c in biota.columns]
if value_cols and "BiotaID" in biota.columns:
    biota["individualCount"] = biota[value_cols].bfill(axis=1).iloc[:, 0]
    biota.to_parquet("cb_long.parquet", index=False)

samplingEffort

In [198]:
biota = pd.read_parquet("cb_long.parquet")
sample_df = sheets.get("Sample", pd.DataFrame())
# Build samplingEffort from Sample sheet values keyed by SampleID
if "SampleID" in biota.columns and "SampleID" in sample_df.columns:
    sample_effort = sample_df[["SampleID", "SampledArea", "ReferenceArea"]].drop_duplicates(subset=["SampleID"])
    biota = biota.merge(sample_effort, on="SampleID", how="left", suffixes=("", "_sample"))

    def format_effort(row):
        parts = []
        if pd.notna(row.get("ReferenceArea")):
            parts.append(f"reference area (m2) {row['ReferenceArea']}")
        if pd.notna(row.get("SampledArea")):
            parts.append(f"area sampled (m2) {row['SampledArea']}")
        return "; ".join(parts) if parts else pd.NA

    biota["samplingEffort"] = biota.apply(format_effort, axis=1).astype("string")
    biota = biota.drop(columns=[c for c in ["SampledArea", "ReferenceArea"] if c in biota.columns])

    biota.to_parquet("cb_long.parquet", index=False)

subSampleToSampleRatio

In [199]:
biota = pd.read_parquet("cb_long.parquet")
subsample_df = sheets.get("Subsample", pd.DataFrame())

if "SubSampleID" in biota.columns and "SubSampleID" in subsample_df.columns and "Ratio" in subsample_df.columns:
    subsample_ratio = subsample_df[["SubSampleID", "Ratio"]].drop_duplicates(subset=["SubSampleID"])
    biota = biota.merge(subsample_ratio, on="SubSampleID", how="left", suffixes=("", "_subsample"))
    if "Ratio_subsample" in biota.columns:
        biota["Ratio"] = biota["Ratio"].fillna(biota["Ratio_subsample"]) if "Ratio" in biota.columns else biota["Ratio_subsample"]
        biota = biota.drop(columns=["Ratio_subsample"])
    biota.to_parquet("cb_long.parquet", index=False)

# Add gebco column

In [200]:
# Step 1: Load the Parquet file
input_file = "cb_long.parquet"
df = pd.read_parquet(input_file)
# Step 2: Extract unique station locations
unique_stations = df[["LatStart", "LonStart"]].drop_duplicates()
# Step 3: Save the result to a new file
output_file = "unique_coordinates.csv"
unique_stations.to_csv(output_file, index=False)
print(f"Unique station locations with year saved to {output_file}")

Unique station locations with year saved to unique_coordinates.csv


In [201]:
def extract_raster_values(raster_path, coordinates_df, batch_size=10000):
    """
    Extract raster values for given coordinates from a single raster file.
    Parameters:
        raster_path (str): Path to the merged raster file.
        coordinates_df (pd.DataFrame): DataFrame with 'lat_start' and 'lon_start'.
        batch_size (int): Number of points to process in each batch.
    Returns:
        pd.DataFrame: DataFrame with extracted raster values.
    """
    results = []
    total_batches = len(coordinates_df) // batch_size + (1 if len(coordinates_df) % batch_size != 0 else 0)
    with rasterio.open(raster_path) as src:
        bounds = src.bounds  # Get raster bounds
        with tqdm.tqdm(total=total_batches, desc="Processing Batches", unit="batch") as pbar:
            for i in range(0, len(coordinates_df), batch_size):
                batch = coordinates_df.iloc[i:i + batch_size].copy()
                batch.loc[:, 'gebco'] = np.nan  # Initialize the column for raster values
                # Adjust longitudes at the antimeridian
                batch['LonStart'] = batch['LonStart'].apply(
                    lambda lon: -180 if lon == 180 else lon
                )
                # Filter coordinates within the raster bounds
                batch_in_bounds = batch[
                    (batch['LonStart'] >= bounds.left) &
                    (batch['LonStart'] <= bounds.right) &
                    (batch['LatStart'] >= bounds.bottom) &
                    (batch['LatStart'] <= bounds.top)
                ]
                # Extract raster values for these coordinates
                coords = [(lon, lat) for lon, lat in zip(batch_in_bounds['LonStart'], batch_in_bounds['LatStart'])]
                values = list(src.sample(coords))
                # Assign values to the batch DataFrame
                for idx, value in zip(batch_in_bounds.index, values):
                    if value[0] is not None and not np.isnan(value[0]):
                        batch.loc[idx, 'gebco'] = value[0]
                results.append(batch)
                pbar.update(1)
    return pd.concat(results, ignore_index=True)
# File path to the merged raster
raster_file = r"..\GEBCO\gebco_merged.tif"
# Load the unique_coordinates.csv file
coordinates = pd.read_csv("unique_coordinates.csv")
# Ensure lon_start and lat_start are numeric
coordinates['LonStart'] = pd.to_numeric(coordinates['LonStart'], errors='coerce')
coordinates['LatStart'] = pd.to_numeric(coordinates['LatStart'], errors='coerce')
# Drop rows with invalid (NaN) coordinates
coordinates = coordinates.dropna(subset=['LonStart', 'LatStart'])
# Extract raster values
result_df = extract_raster_values(raster_file, coordinates)
# Save the results to a CSV file
result_df.to_csv("gebco_values.csv", index=False)
print("Results saved as'gebco_values.csv'.")

Processing Batches: 100%|██████████| 2/2 [00:01<00:00,  1.99batch/s]

Results saved as'gebco_values.csv'.


In [202]:
gebco_df = pd.read_csv("gebco_values.csv")
cb = pd.read_parquet("cb_long.parquet")

cb = cb.merge(
    gebco_df.rename(columns={"gebco": "gebcoDepth"}),
    on=["LatStart", "LonStart"],
    how="left"
)

cb.to_parquet("cb_long.parquet")
print("Updated cb_long.parquet with gebcoDepth.")

Updated cb_long.parquet with gebcoDepth.


# Cut according to Arctic LMEs

In [203]:


cb = pd.read_parquet("cb_long.parquet")

# Load LME boundaries
lme_path = r"..\\LME_boundaries_2014\\LME_2013_polygon_projected.shp"
lme = gpd.read_file(lme_path).to_crs("EPSG:4326")

# Build points from coordinates and filter to within LME polygons
cb_gdf = gpd.GeoDataFrame(
    cb,
    geometry=gpd.points_from_xy(cb["LonStart"], cb["LatStart"]),
    crs="EPSG:4326",
)

lme_union = lme.union_all()
cb_in_lme = cb_gdf[cb_gdf.geometry.within(lme_union)].drop(columns=["geometry"])

cb_in_lme.to_parquet("cb_long_LME.parquet", index=False)
print(f"Filtered cb_long.parquet to {len(cb_in_lme):,} rows within LME boundaries.")

Filtered cb_long.parquet to 243,722 rows within LME boundaries.


# Map to final file

In [204]:
cb = pd.read_parquet("cb_long_LME.parquet")

column_map = {
    "sourceID": "BiotaID",
    "datasetKey": "Project",
    "datasetTitle": "Title",
    "decimalLatitudeStart": "LatStart",
    "decimalLongitudeStart": "LonStart",
    "decimalLatitudeStop": "LatEnd",
    "decimalLongitudeStop": "LonEnd",
    "depthA": "DepthStart",
    "depthB": "DepthEnd",
    "gebcoDepth": "gebcoDepth",
    "eventID": "SubsetID",
    "waterbody": "Realm",
    "eventDate": "DateStart",
    "endDayOfYear": "DateEnd", 
    "year": "year",
    "eventRemarks": "SampleRemark",
    "verbatimScientificName": "ScientificName (User)",
    "acceptedScientificName": "ScientificName (Accepted)",
    "identificationQualifier": "Specification",
    "taxonomicStatus": "Status",
    "taxonomicIssue": "UnacceptReason",
    "taxonRank": "Rank",
    "phylum": "Phylum",
    "class": "Class",
    "order": "Order",
    "family": "Family",
    "genus": "Genus",
    "validAuthority": "ValidAuthority",
    "aphiaID": "AphiaId (Accepted)",
    "individualCount": "individualCount",
    "samplingEffort": "samplingEffort",
    "biomass": "SumWet",
    "occurrenceRemarks": "Remark",
    "gear": "Gear",
    "gearCategory": "Category",
    "gearDescription": "Description",
    "meshSize": "MeshSize",
    "sieveFraction": "SieveFraction",
    "samplingEffort": "samplingEffort",
    "subsampleToSampleRationo": "Ratio",
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeoSpatialIssues",
    "depthA",
    "depthAType",
    "depthB",
    "depthBType",
    "depthAccuracy",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "eventDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "samplingEffort",
    "percentCoverage",
    "biomass",
    "biomassUnit",
    "dryMass",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "gearDescription",
    "gearDimensions",
    "meshSize",
    "sieveFraction",
    "sampleSizeValue",
    "sampleSizeUnit",
    "subsampleToSampleRatio",
    "subsampleSpatialLevel",
    "flagged", 
    "flaggedComment"
]

cb_final = pd.DataFrame(index=cb.index)
missing_sources = []

for col in output_columns:
    if col in column_map:
        src = column_map[col]
        if src in cb.columns:
            cb_final[col] = cb[src]
        else:
            cb_final[col] = pd.NA
            missing_sources.append(src)
    else:
        cb_final[col] = pd.NA

cb_final = cb_final[output_columns]
cb_final.to_parquet("cb_final.parquet", index=False)

print(f"Saved cb_final.parquet with {len(cb_final):,} rows and {len(cb_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in cb:", sorted(set(missing_sources)))

Saved cb_final.parquet with 243,722 rows and 67 columns.


In [205]:
cb_final = pd.read_parquet("cb_final.parquet")
cb_final["sourceDatabase"] = "CRITTERBASE"
cb_final.to_parquet("cb_final.parquet", index=False)
print("Added sourceDatabase column with value 'CRITTERBASE' and saved cb_final.parquet.")

Added sourceDatabase column with value 'CRITTERBASE' and saved cb_final.parquet.
